# Logistic Regression — Comprehensive Cheat Sheet

Covers theory, assumptions, scikit-learn API, evaluation metrics, regularisation, and practical tips.

---
## 1. Core Formula

Logistic regression models the probability that $y = 1$ given features $\mathbf{x}$:

$$P(y=1 \mid \mathbf{x}) = \sigma(\beta_0 + \beta_1 x_1 + \dots + \beta_k x_k)$$

where $\sigma(z) = \frac{1}{1+e^{-z}}$ is the sigmoid function.

- **Odds** = $\frac{p}{1-p}$
- **Log-odds (logit)** = $\ln\frac{p}{1-p} = \beta_0 + \sum \beta_j x_j$
- A one-unit increase in $x_j$ multiplies the odds by $e^{\beta_j}$.


---
## 2. Key Assumptions

| # | Assumption | How to Check |
|---|---|---|
| 1 | Binary (or ordinal) outcome | `df['target'].unique()` |
| 2 | Linear relationship between logit and continuous predictors | Boxplots, partial residual plots |
| 3 | No multicollinearity among features | VIF, correlation heatmap |
| 4 | Independence of observations | Study design ensures this |
| 5 | Large sample size (rule of thumb ≥10 events per predictor) | `len(y[y==1]) / X.shape[1]` |
| 6 | No influential outliers | Cook's distance, leverage plots |

---
## 3. scikit-learn API Quick Reference

```python
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(
    penalty='l2',           # 'l1', 'l2', 'elasticnet', None
    C=1.0,                  # inverse of regularization strength; smaller = stronger
    solver='lbfgs',         # 'lbfgs'(default, L2), 'liblinear'(L1/L2), 'saga'(L1/L2/elasticnet)
    class_weight=None,      # None or 'balanced'
    max_iter=100,            # increase if convergence warning
    random_state=None,
    n_jobs=None,             # parallelise if solver supports it
)
model.fit(X_train, y_train)
model.predict(X_test)        # class labels (0/1)
model.predict_proba(X_test)  # [[prob_class0, prob_class1], ...]
model.decision_function(X_test)  # raw scores (logit)
model.score(X_test, y_test)  # accuracy
model.intercept_, model.coef_  # parameters
```

---
## 4. Solver-Penalty Compatibility

| Solver | L1 | L2 | ElasticNet | No Penalty |
|--------|----|----|-----------|------------|
| lbfgs | ❌ | ✅ | ❌ | ✅ |
| liblinear | ✅ | ✅ | ❌ | ❌ |
| newton-cg | ❌ | ✅ | ❌ | ✅ |
| sag | ❌ | ✅ | ❌ | ✅ |
| saga | ✅ | ✅ | ✅ | ✅ |

**Use liblinear for small datasets. Use saga for large datasets or elasticnet.**

---
## 5. Regularisation — What & Why

### L1 (Lasso)
- Penalty: $\lambda \sum |\beta_j|$
- Drives some coefficients to exactly zero → built-in feature selection
- Good when you suspect many irrelevant features

### L2 (Ridge)
- Penalty: $\lambda \sum \beta_j^2$
- Shrinks coefficients toward zero but rarely to exactly zero
- Good when multicollinearity exists

### ElasticNet
- Hybrid of L1 and L2
- Controlled by `l1_ratio` (0 = pure L2, 1 = pure L1)

**C parameter**: `C = 1/λ`. Smaller C → stronger regularisation.


---
## 6. Evaluation Metrics — Decision Guide

```
                    Predicted 0    Predicted 1
Actual 0              TN              FP
Actual 1              FN              TP
```

| Metric | Formula | When to Use |
|--------|---------|------|
| Accuracy | (TP+TN)/Total | Classes balanced |
| Precision | TP/(TP+FP) | Cost of false positives high (e.g. spam filter) |
| Recall | TP/(TP+FN) | Cost of false negatives high (e.g. disease screening) |
| F1 | 2·P·R/(P+R) | Balance precision & recall |
| ROC-AUC | Area under TPR vs FPR | Overall ranking quality |
| PR-AUC | Area under Precision-Recall | Highly imbalanced datasets |

**Rule of thumb**: ROC-AUC ≥ 0.9 = excellent, 0.8–0.9 = good, 0.7–0.8 = acceptable, <0.7 = poor.

---
## 7. Handling Class Imbalance — Options

| Method | Pros | Cons |
|--------|------|------|
| `class_weight='balanced'` | Simple, no data change | Overweights noisy minority samples |
| SMOTE / ADASYN | Creates synthetic minority samples | May create unrealistic samples |
| Random Undersampling | Reduces training time | Loses majority-class information |
| Threshold tuning | Free post-hoc fix | Must re-evaluate on validation set |

---
## 8. Feature Preprocessing Checklist

| Data Type | Preprocessing | Code |
|-----------|--------------|------|
| Continuous | Scale (StandardScaler / MinMax) | `StandardScaler()` |
| Highly skewed | Log transform | `np.log1p()` |
| Categorical (nominal) | One-hot encode | `pd.get_dummies()` or `OneHotEncoder()` |
| Categorical (ordinal) | Ordinal encode or one-hot | `OrdinalEncoder()` |
| Missing values | Impute mean/median/mode | `SimpleImputer()` |

---
## 9. Common Pitfalls & Fixes

| Pitfall | Symptom | Fix |
|---------|---------|-----|
| ConvergenceWarning | Model doesn't converge | Increase `max_iter`, scale features |
| Leakage | Test accuracy >> train accuracy | Scale within pipeline, split first |
| Dummy trap | Collinear features inflate variance | `drop_first=True` in get_dummies |
| Wrong solver | ValueError on penalty | Match solver to penalty (see table above) |
| Interpretation error | Confusing sign of coefficient | Check dummy encoding reference category |
| Threshold default | Optimistic accuracy on imbalanced data | Tune threshold on validation set |
| Data snooping | Inflated metrics | Keep test set truly held-out until final eval |

---
## 10. Interpretation Cheat Codes

```
coef > 0 → positive association with y=1
coef < 0 → negative association with y=1
|coef| large → strong effect (but only comparable if features scaled!)

Odds ratio for feature j:
    OR_j = exp(beta_j)
    OR > 1 → higher odds of positive class
    OR < 1 → lower odds

If beta_j = 0.5 → OR ≈ 1.65 → 65% increase in odds per unit increase in x_j
If beta_j = -0.5 → OR ≈ 0.61 → 39% decrease in odds
```

```python
import numpy as np
odds_ratios = np.exp(model.coef_[0])
for var, or_val in zip(X_train.columns, odds_ratios):
    print(f'{var:30s}  OR = {or_val:.3f}')
```

---
## 11. Pipeline Template (Production-Safe)

```python
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

numeric_pipe = Pipeline([
    ('impute', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])

categorical_pipe = Pipeline([
    ('impute', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(drop='first', handle_unknown='ignore')),
])

preprocessor = ColumnTransformer([
    ('num', numeric_pipe, numeric_features),
    ('cat', categorical_pipe, categorical_features),
])

full_pipeline = Pipeline([
    ('preprocess', preprocessor),
    ('classifier', LogisticRegression(C=0.05, penalty='l1',
                                       solver='liblinear', max_iter=2000)),
])

full_pipeline.fit(X_raw, y)
```

Key benefit: preprocessing is **fitted only on training data** inside the pipeline during `fit()`, preventing data leakage.

---
## 12. Cross-Validation Patterns

```python
# Basic 5-fold
from sklearn.model_selection import cross_val_score
scores = cross_val_score(model, X, y, cv=5, scoring='roc_auc')

# Stratified (preserves class ratio)
from sklearn.model_selection import StratifiedKFold
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(model, X, y, cv=cv, scoring='f1')

# Repeated stratified (more robust estimate)
from sklearn.model_selection import RepeatedStratifiedKFold
rcv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=42)
scores = cross_val_score(model, X, y, cv=rcv, scoring='roc_auc')
```


---
## 13. Quick Diagnostic Snippets

```python
# Events per variable (EPV) — should be ≥10
epv = min(y.sum(), (y == 0).sum()) / X.shape[1]
print(f'EPV = {epv:.1f}')

# Variance Inflation Factor
from statsmodels.stats.outliers_influence import variance_inflation_factor
vif = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
# Rule: VIF > 5 suggests multicollinearity

# McFadden pseudo R² (needs statsmodels)
# import statsmodels.api as sm
# ll_full = sm.Logit(y_train, sm.add_constant(X_train)).fit(disp=0).llf
# ll_null = sm.Logit(y_train, sm.add_constant(np.ones(len(y_train)))).fit(disp=0).llf
# pseudo_r2 = 1 - ll_full / ll_null
```